In [1]:
import os, math, re

import numpy as np
import pandas as pd
from nilearn.connectome import ConnectivityMeasure

import matplotlib.pyplot as plt
import seaborn as sns

import pingouin as pg

In [ ]:
# ---------------- user settings ----------------
atlas = 'difumo'
OUTPUT_DIR   = '/nfs/roberts/pi/pi_il77/Nachshon/PSUB/preProc_o/corMatrix_byTrial_pr/'
TR_SEC       = 1.0
HOT_DUR_SEC  = 60
MIN_TRS_FULL = 60     
MIN_TRS_HOT  = 40     
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Map FD 'file' -> canonical Run ("run-1"/"run-2"/"run-3")
def file_to_run(s):
    m = re.search(r'run-(\d+)', str(s))
    return f'run-{m.group(1)}' if m else 'run-1'   # "traumarecall" -> "run-1"

In [ ]:
behav_data = pd.read_csv('/home/nk549/Documents/PSUB/BehavioralData/clinical.csv')

PTSD = behav_data[behav_data['group']=='PTSD']['sub_id'].values
HC = behav_data[behav_data['group']=='HC']['sub_id'].values
print(f"number of PTSD participants {len(PTSD)}\nNumber of HC participants {len(HC)}")

In [ ]:
FD_file = pd.read_csv('/nfs/roberts/pi/pi_il77/Nachshon/PSUB/trauma_recall_FD.csv')

fd_threshold = .5
# --- Normalize keys ---
# sub: "sub-1566" -> sub_id: 1566
FD_file['run']    = FD_file['file'].apply(file_to_run)

# Ensure behav keys align
behav_data['sub_id'] = behav_data['sub_id'].astype(int)
behav_data['run'] = behav_data['run'].astype(str).str.lower().fillna('run-1')

# --- Merge FD onto behavior by (sub_id, Run) ---
merged = behav_data.merge(
    FD_file[['sub_id', 'run', 'FD']],
    on=['sub_id', 'run'], how='left', validate='m:1'
)

# --- Report FD > 0.5 (per run) ---
high_fd = merged.loc[merged['FD'].notna() & (merged['FD'] > fd_threshold), ['sub_id', 'run', 'FD', 'group']]
for _, r in high_fd.iterrows():
    print(f"High FD: sub-{int(r.sub_id)} {r.group} {r.run} -> {r.FD:.3f}")

# --- Remove rows with FD > 0.5 from behav_data ---
behav_data_filtered = merged.loc[~(merged['FD'] > fd_threshold)].drop(columns=['FD']).copy()

print(f"Rows in behav_data: {len(behav_data)} | after per-run FD<={fd_threshold} filter: {len(behav_data_filtered)}")

In [ ]:
ts_files = []
hot_spot = {}

for _, row in behav_data_filtered.iterrows():
    sub   = str(row['sub_id'])
    run   = str(row['run'])
    onset = row.get('onset', np.nan)

    # skip subjects without a hotspot
    if pd.isna(onset):
        continue

    hot_spot[sub] = float(onset)  # seconds
    p = f"/nfs/roberts/pi/pi_il77/Nachshon/PSUB/preProc_o/sub-{sub}_trauma_{run}_Average_ROI_difumo.csv"
    if os.path.exists(p):
        ts_files.append(p)

In [ ]:
valid_ids = [str(k).strip() for k in hot_spot.keys()]
filtered_behav_data = behav_data[behav_data['sub_id'].astype(str).isin(valid_ids)].copy()

# 1. Run the t-test and store the result
res = pg.ttest(
    x=filtered_behav_data[filtered_behav_data['group']=='HC']['onset'],
    y=filtered_behav_data[filtered_behav_data['group']=='PTSD']['onset']
)

# 2. Extract values for the label
p_val = res['p_val'].values[0]
bf10 = res['BF10'].values[0]
ci_lower = res['CI95'].values[0][0]
ci_upper = res['CI95'].values[0][1]

stats_str = (
    f"p = {p_val:.3f}\n"
    f"95% CI = [{ci_lower:.2f}, {ci_upper:.2f}]\n"
    f"BF10 = {float(bf10):.3f}"
)

# 3. Create the plot
plt.figure(figsize=(10, 6))
sns.set_theme(style="white") # Cleaner for publications

ax = sns.histplot(
    data=filtered_behav_data, 
    x='onset', 
    hue='group', 
    kde=True, 
    palette='viridis',
    alpha=0.4
)

# 4. Add  overlay text
plt.text(
    0.95, 0.95, stats_str, 
    transform=ax.transAxes, 
    fontsize=10, 
    verticalalignment='top', 
    horizontalalignment='right',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='lightgrey')
)

plt.title('Onset Distribution: HC vs PTSD')

# 5. Save as SVG
plt.savefig('onset_stats_plot.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
# nilearn connectivity
connectome_measure = ConnectivityMeasure(kind='correlation', standardize='zscore_sample')

In [ ]:
def sec_to_idx(onset_s: float, duration_s: float, T: int, tr: float = 1.0):
    # Map [onset, onset+duration) seconds to [start, end) row indices, clipped to [0, T)
    start = int(math.floor(onset_s / tr))
    end   = int(math.ceil((onset_s + duration_s) / tr))
    start = max(0, start); end = min(T, end)
    return start, max(start, end)

def fisher_pool_r_mats(r_mats, lengths):
    """
    r_mats: list of (p,p) correlation matrices for trials in a condition
    lengths: list of segment lengths (T_b), used to weight by (T_b-3)
    Returns pooled r via Fisher-z average (length-weighted).
    """
    eps = 1e-7
    z_mats = [np.arctanh(np.clip(R, -1+eps, 1-eps)) for R in r_mats]
    w      = np.array([max(L-3, 1) for L in lengths], dtype=float)  # guard tiny segments
    z_bar  = np.average(np.stack(z_mats, axis=0), axis=0, weights=w)
    return np.tanh(z_bar)

In [ ]:
def sec_to_idx_bounds(onset_sec, dur_sec, T, tr=1.0):
    i0 = int(np.floor(onset_sec / tr))
    i1 = int(np.ceil((onset_sec + dur_sec) / tr))
    i0 = max(0, min(i0, T))
    i1 = max(0, min(i1, T))
    return i0, i1


In [ ]:
import hashlib
seed = int(hashlib.md5(sub.encode()).hexdigest(), 16) % 10**8
np.random.seed(seed)

In [ ]:
overlap_list = []

for ts_path in ts_files:
    base = os.path.basename(ts_path)
    # Extract sub and run from filename
    sub  = base.split('_')[0].split('-')[-1]
    run = base.split('_trauma_')[1].split('_')[0].replace("run-","")

    # ---- 1. Load Time Series (rows=time, cols=ROIs) ----
    ts_df    = pd.read_csv(ts_path)
    # Ensure we only use numeric ROI columns
    roi_cols = [c for c in ts_df.columns if np.issubdtype(ts_df[c].dtype, np.number)]
    ts       = ts_df[roi_cols].to_numpy()
    T, p     = ts.shape
    
    # Atlas validation
    if atlas == 'difumo':
        assert p == 256, f"Expected 256 ROIs, got {p} in {ts_path}"
    elif atlas == 'shen':
        assert p == 268, f"Expected 268 ROIs, got {p} in {ts_path}"

    # ---- 2. FULL-RUN FC ----
    if T >= MIN_TRS_FULL:
        cor_full = connectome_measure.fit_transform([ts])[0]
        df_full  = pd.DataFrame(cor_full, columns=roi_cols)
        df_full.insert(0, "roi", np.arange(p))
        out_full = os.path.join(OUTPUT_DIR, f"sub-{sub}_run-{run}_FULL_{atlas}.csv")
        df_full.to_csv(out_full, index=False)
        print(f"[sub-{sub} run-{run}] FULL: T={T} → {out_full}")
    else:
        print(f"[sub-{sub} run-{run}] FULL: skip (T={T} < {MIN_TRS_FULL})")

    # ---- 3. HOTSPOT FC (60s from hot_spot[sub]) ----
    # Safety check for missing sub_id in dictionary
    raw_onset = hot_spot.get(sub, None)
    
    if raw_onset is None:
        print(f"[sub-{sub} run-{run}] HOT60: no hotspot onset found → skipping HOT/RAND")
        continue
    
    hs_onset = raw_onset-4
    i0, i1 = sec_to_idx_bounds(hs_onset, HOT_DUR_SEC, T, TR_SEC)
    
    if (i1 - i0) >= MIN_TRS_HOT:
        cor_hot = connectome_measure.fit_transform([ts[i0:i1, :]])[0]
        df_hot  = pd.DataFrame(cor_hot, columns=roi_cols)
        df_hot.insert(0, "roi", np.arange(p))
        out_hot = os.path.join(OUTPUT_DIR, f"sub-{sub}_run-{run}_HOT60_{atlas}.csv")
        df_hot.to_csv(out_hot, index=False)
        print(f"[sub-{sub} run-{run}] HOT60: {i0}:{i1} (len={i1-i0}) → {out_hot}")
    else:
        print(f"[sub-{sub} run-{run}] HOT60: window {i0}:{i1} too short → skip")

    # ---- 4. RANDOM 60s FC (Finding Minimum Overlap) ----
    seed_str = f"{sub}_{run}"
    seed = int(hashlib.md5(seed_str.encode()).hexdigest(), 16) % 10**8
    rng = np.random.default_rng(seed)
    
    max_onset_sec = (T * TR_SEC) - HOT_DUR_SEC
    
    if max_onset_sec > 0.0:
        best_overlap = float('inf')
        best_indices = (0, 0)
        best_onset = 0
        for attempt in range(100):
            tmp_onset = rng.uniform(0.0, max_onset_sec)
            r0, r1 = sec_to_idx_bounds(tmp_onset, HOT_DUR_SEC, T, TR_SEC)
            
            # Intersection formula: max(0, min_end - max_start)
            current_overlap = max(0, min(r1, i1) - max(r0, i0))
            
            # Keep the window with the lowest overlap
            if current_overlap < best_overlap:
                best_overlap = current_overlap
                best_indices = (r0, r1)
                best_onset = tmp_onset
            
            # If we hit 0, we can stop early
            if best_overlap == 0:
                break
        
        # Use the best window found
        r0, r1 = best_indices
        # Store overlap for histogram later
        overlap_list.append(best_overlap) 
        if (r1 - r0) >= MIN_TRS_HOT:
            cor_rand = connectome_measure.fit_transform([ts[r0:r1, :]])[0]
            df_rand  = pd.DataFrame(cor_rand, columns=roi_cols)
            df_rand.insert(0, "roi", np.arange(p))
            out_rand = os.path.join(OUTPUT_DIR, f"sub-{sub}_run-{run}_RANDOM60_{atlas}.csv")
            df_rand.to_csv(out_rand, index=False)
            print(f"[sub-{sub} run-{run}] RAND60: {r0}:{r1} (Min Overlap={best_overlap} TRs) → {out_rand}")
        else:
            print(f"[sub-{sub} run-{run}] RAND60: window too short → skip")
    else:
        print(f"[sub-{sub} run-{run}] RAND60: scan too short for 60s window → skip")
    

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(overlap_list, bins=15, kde=False, color='salmon')

plt.title('Distribution of Overlap between Hotspot and Random Windows')
plt.xlabel('Overlap (Number of TRs)')
plt.ylabel('Number of Participants')
plt.axvline(0, color='black', linestyle='--') # Mark the ideal 0 overlap
plt.grid(axis='y', alpha=0.3)

plt.savefig('overlap_histogram.svg')
plt.show()